# Medical QSVM: understanding the CPU and MerLin adaptations

This notebook presents the small results that have already been computed. It does not rerun a quantum simulation or kernel calculation.

Three scopes must remain distinct:

1. the reference reproduction using the paper's controlled medical embeddings, which is not performed here;
2. the surrogate study using PneumoniaMNIST pixels;
3. the MerLin photonic adaptation, which is not the paper's qubit BSP circuit.

## 1. Load the curated artifacts

The files under `results/` contain only small metrics and metadata. The per-seed artifact includes a dataset checksum and split identifiers so that comparisons can be checked. Images, pixels, kernel matrices, and private paths are not included.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
results_path = project_root / "results"
per_seed_path = results_path / "q4_n100_per_seed.csv"
merlin_path = results_path / "merlin_q2_seed0.json"

if not per_seed_path.exists() or not merlin_path.exists():
    raise FileNotFoundError("Run this notebook from the qsvm_medimage root.")

per_seed_results = pd.read_csv(per_seed_path)
per_seed_results.head()

## 2. CPU comparison at q=4

The primary metric is the **minority-class F1** (`normal`). It combines:

- precision: among images predicted as normal, how many are actually normal;
- recall: among genuinely normal images, how many are detected.

A minority-class F1 of zero means that no minority example was correctly detected.

In [ ]:
q4 = per_seed_results.query(
    "scope == 'open_data_surrogate' and pca_dim == 4 and n_samples == 100"
).copy()
comparison_keys = ["model", "pca_dim", "data_seed"]
if q4.duplicated(comparison_keys).any():
    raise ValueError("Duplicate q=4 configurations found in the curated artifact.")

model_order = ["qsvm", "linear_svm", "rbf_svm"]
expected_pairs = {(model, seed) for model in model_order for seed in range(10)}
observed_pairs = set(q4[["model", "data_seed"]].itertuples(index=False, name=None))
if observed_pairs != expected_pairs:
    raise ValueError("The curated artifact is incomplete or contains unexpected runs.")
if not q4.groupby("data_seed")["split_id"].nunique().eq(1).all():
    raise ValueError("Compared models do not use the same split for every seed.")

model_labels = {
    "qsvm": "QSVM",
    "linear_svm": "Linear SVM",
    "rbf_svm": "Tuned RBF SVM",
}
summary = (
    q4.assign(zero_f1=q4["test_minority_f1"].eq(0))
    .groupby("model", as_index=False)
    .agg(
        n_seeds=("data_seed", "nunique"),
        mean_test_minority_f1=("test_minority_f1", "mean"),
        std_test_minority_f1=("test_minority_f1", "std"),
        zero_f1_seeds=("zero_f1", "sum"),
    )
)
summary["model_label"] = summary["model"].map(model_labels)
summary = summary.set_index("model").loc[model_order].reset_index()
summary[["model_label", "n_seeds", "mean_test_minority_f1", "std_test_minority_f1", "zero_f1_seeds"]]

In [ ]:
paired = q4.pivot(
    index="data_seed", columns="model", values="test_minority_f1"
).sort_index()
paired_deltas = pd.DataFrame(
    {
        "QSVM - linear": paired["qsvm"] - paired["linear_svm"],
        "QSVM - tuned RBF": paired["qsvm"] - paired["rbf_svm"],
    }
)
tolerance = 1e-12
paired_summary = pd.DataFrame(
    [
        {
            "comparison": comparison,
            "mean_delta": delta.mean(),
            "wins": int((delta > tolerance).sum()),
            "ties": int((delta.abs() <= tolerance).sum()),
            "losses": int((delta < -tolerance).sum()),
        }
        for comparison, delta in paired_deltas.items()
    ]
)
rbf_c_counts = (
    q4.query("model == 'rbf_svm'")["svc_c"]
    .value_counts()
    .sort_index()
    .rename_axis("selected_c")
    .to_frame("seeds")
)
display(paired_summary)
display(rbf_c_counts)

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for comparison in paired_deltas:
    axes[0].plot(
        paired_deltas.index, paired_deltas[comparison], marker="o", label=comparison
    )
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_xlabel("Data and split seed")
axes[0].set_ylabel("Paired minority-class F1 difference")
axes[0].set_title("Paired differences at q=4")
axes[0].legend()

axes[1].bar(
    summary["model_label"], summary["zero_f1_seeds"] / summary["n_seeds"]
)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Fraction of seeds with minority F1 = 0")
axes[1].set_title("Minority-class collapse at q=4")
axes[1].tick_params(axis="x", rotation=15)
figure.suptitle("PneumoniaMNIST, q=4, N=100, 10 paired seeds")
figure.tight_layout()

The QSVM mean is slightly above the linear SVM (`+0.013`) but slightly below the tuned RBF (`-0.020`). For both paired comparisons, QSVM has one win, eight ties, and one loss. All three models have a minority-class F1 of zero on 2 of the 10 seeds. The RBF validation procedure selects `C=0.1` once, `C=1` five times, `C=10` three times, and `C=100` once.

Each test split contains only 2--4 minority examples, so these differences are exploratory. Here, a seed controls both subsampling and the train/validation/test split; it is not an embedding-generation seed as in the paper. This surrogate study does not reproduce the paper's systematic QSVM advantage.

## 3. MerLin photonic-kernel smoke

This experiment uses two PCA components, three optical modes, and the Fock state `[1, 0, 1]`. The data seed and circuit seed are both zero, but they have distinct roles.

In [ ]:
with merlin_path.open() as file:
    merlin = json.load(file)

merlin_metrics = pd.DataFrame(merlin["metrics"]).T
merlin_metrics.index.name = "split"
merlin_metrics

In [ ]:
ax = merlin_metrics[["accuracy", "minority_f1", "auc"]].plot.bar(
    figsize=(8, 4), rot=0
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("MerLin: PCA=2, modes=3, data seed=0, circuit seed=0")
ax.figure.tight_layout()

The test AUC is 1, while the minority-class F1 is 0. This is not contradictory: AUC measures score ranking across all possible thresholds, whereas F1 uses the decisions produced at the SVC threshold. Here, the model ranks the scores correctly but ultimately predicts every image as `pneumonia`.

The test split contains only ten images, including two normal images. This result validates that MerLin runs technically on CPU; it does not demonstrate a photonic advantage.

## 4. What remains to be established

- PneumoniaMNIST pixels and the paper's frozen embeddings are not scientifically equivalent.
- The MerLin kernel and the qubit BSP circuit are different feature maps.
- The complete q=4, N=100 per-seed artifact validates pairing and run completeness, but its small test splits do not support a quantum-advantage claim.
- A larger N=500 surrogate experiment is the next empirical check.
- Every subsequent comparison must use the same samples and splits for all compared models.